In [1]:
import os
# Force the system to only see GPU #1. (Change to 0, 2, 3 etc. as needed)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from datasets import load_dataset

/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # normalized float 4 — better accuracy than plain int4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # extra quantization of the quantization constants, saves a bit more memory
)

In [4]:
model = AutoModelForImageTextToText.from_pretrained(
    "OpenGVLab/InternVL3_5-38B-HF",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 2/1392 [00:00<07:09,  3.24it/s]/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 1392/1392 [00:16<00:00, 83.85it/s] 
[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [5]:
processor = AutoProcessor.from_pretrained("OpenGVLab/InternVL3_5-38B-HF")

### Testing Quilt-VQA

In [6]:
quilt_vqa_dataset = load_dataset('wisdomik/Quilt_VQA')

In [7]:
quilt_vqa_dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer', 'answer_type', 'context'],
        num_rows: 985
    })
})

### Single Message Testing

In [8]:
data_idx = 0
data_inst = quilt_vqa_dataset['train'][data_idx]
data_inst

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1920x1080>,
 'question': 'What is the visual characteristic of the cells after therapy?',
 'answer': 'After therapy, the cells are following the path of neuroendocrine differentiation and forming ganglion cells with big prominent cell nuclei and abundant eosinophilic cytoplasm.',
 'answer_type': 'OPEN',
 'context': " This is differentiation following therapy, and of course, this is the ultimate form of differentiation following therapy. It's following that path of neuroendocrine differentiation and this time around forming ganglion cells with those big prominent cell nuclei and that abundant eosinophilic cytoplasm. Isn't that pretty? If there was only one marker that you were allowed to use to make a diagnosis of Ewing's, it would be CD99. What you're looking for is diffuse, virtually all of the cells should be positive, positivity for CD99."}

In [9]:
R1_SYSTEM_PROMPT = """
You are an AI assistant that rigorously follows this response protocol:

1. First, conduct a detailed analysis of the question. Consider different angles, potential solutions, and reason through the problem step-by-step. Enclose this entire thinking process within <think> and </think> tags.

2. After the thinking section, provide a clear, concise, and direct answer to the user's question. Separate the answer from the think section with a newline.

Ensure that the thinking process is thorough but remains focused on the query. The final answer should be standalone and not reference the thinking section.
""".strip()

In [10]:
messages = [
        {"role": "system", "content": R1_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": data_inst['image']},
                {"type": "text", "text": data_inst['question']},
            ],
        },
    ]

In [11]:
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [12]:
text

"<|im_start|>system\nYou are an AI assistant that rigorously follows this response protocol:\n\n1. First, conduct a detailed analysis of the question. Consider different angles, potential solutions, and reason through the problem step-by-step. Enclose this entire thinking process within <think> and </think> tags.\n\n2. After the thinking section, provide a clear, concise, and direct answer to the user's question. Separate the answer from the think section with a newline.\n\nEnsure that the thinking process is thorough but remains focused on the query. The final answer should be standalone and not reference the thinking section.<|im_end|>\n<|im_start|>user\n<IMG_CONTEXT>\nWhat is the visual characteristic of the cells after therapy?<|im_end|>\n<|im_start|>assistant\n"

In [13]:
inputs = processor(
    text=[text],
    images=[data_inst['image']],
    padding=True,
    return_tensors="pt",
).to(model.device)

In [14]:
inputs

{'input_ids': tensor([[151644,   8948,    198,  ..., 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'pixel_values': tensor([[[[ 1.6324,  1.7523,  1.8893,  ...,  1.7694,  1.8208,  1.8722],
          [ 1.7009,  1.6153,  1.6153,  ...,  1.9235,  2.0434,  2.0948],
          [ 1.5810,  1.3413,  1.2385,  ...,  1.9407,  2.0263,  1.9920],
          ...,
          [ 2.0434,  2.0777,  2.1119,  ...,  0.8447,  1.0331,  1.0844],
          [ 2.0092,  2.0434,  2.0777,  ...,  0.6906,  0.9474,  1.0502],
          [ 1.9920,  2.0092,  2.0434,  ...,  0.5536,  0.7248,  0.8104]],

         [[ 1.7458,  1.8859,  2.0434,  ...,  1.4307,  1.4482,  1.4657],
          [ 1.7983,  1.7458,  1.7808,  ...,  1.6057,  1.6933,  1.7108],
          [ 1.7108,  1.5007,  1.4132,  ...,  1.6583,  1.7283,  1.6583],
          ...,
          [ 2.3585,  2.3936,  2.4286,  ..., -0.6527, -0.5126, -0.4076],
          [ 2.3235,  2.3585,  2.3936,  ..., -0.7927, -0.6

In [15]:
generated_ids = model.generate(**inputs,
                                max_new_tokens=1024,
                                do_sample=True,
                                temperature=0.6)

/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [19]:
output_text = processor.batch_decode(
    generated_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)

In [20]:
print(output_text[0])

The cells are mostly pink and have a faint purple center.


### Batch Outputs

In [21]:
batch_size = 4
batch_data = quilt_vqa_dataset['train'][0:batch_size]
batch_images = [item.convert("RGB") for item in batch_data['image']]
batch_prompts = [item for item in batch_data['question']]

In [22]:
messages_list = [
    [{"role": "system", "content": R1_SYSTEM_PROMPT},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": prompt},
        ],
    }]
    for img, prompt in zip(batch_images, batch_prompts)
]

In [23]:
messages_list

[[{'role': 'system',
   'content': "You are an AI assistant that rigorously follows this response protocol:\n\n1. First, conduct a detailed analysis of the question. Consider different angles, potential solutions, and reason through the problem step-by-step. Enclose this entire thinking process within <think> and </think> tags.\n\n2. After the thinking section, provide a clear, concise, and direct answer to the user's question. Separate the answer from the think section with a newline.\n\nEnsure that the thinking process is thorough but remains focused on the query. The final answer should be standalone and not reference the thinking section."},
  {'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=1920x1080>},
    {'type': 'text',
     'text': 'What is the visual characteristic of the cells after therapy?'}]}],
 [{'role': 'system',
   'content': "You are an AI assistant that rigorously follows this response protocol:\n\n1. First, conduc

In [24]:
# Apply chat template to each
batch_texts = [
    processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
    for m in messages_list
]

In [25]:
inputs = processor(
    text=batch_texts,
    images=batch_images,
    padding=True,
    return_tensors="pt",
).to(model.device)

In [26]:
generated_ids = model.generate(
    **inputs,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.6,
)

In [27]:
outputs = processor.batch_decode(
    generated_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)

In [28]:
for prompt, output in zip(batch_prompts, outputs):
    print(f"Prompt: {prompt}\nOutput: {output}\n")

Prompt: What is the visual characteristic of the cells after therapy?
Output: The cells appear flat and the nuclei are small.

Prompt: What is the initial hint that the image does not depict Ewing's tumors?
Output: The initial hint that the image does not depict Ewing's tumors is the presence of a complex, colorful background filled with various patterns and symbols, such as hearts, stars, and the letters 'CD99'. This background is not typical of medical imaging or pathology slides, which usually focus on clear, detailed views of tissue samples or anatomical structures. The inclusion of playful and decorative elements suggests that the image is not a clinical representation of Ewing's tumors.

Prompt: What are the signs of aberrant maturation, hyperkeratosis, and acanthosis in the image?
Output: <think>
Okay, so I need to figure out what aberrant maturation, hyperkeratosis, and acanthosis look like in the image provided. Let me start by recalling what each of these terms means.

Aberra